# Process Video — Pipeline Playground

เรียก logic การ process video ของ worker (`worker_app.pipeline.*`) ตรงๆ — ไม่ผ่าน Celery / Mongo / Redis

ใช้สำหรับ:
- ปรับ threshold (audio delta, pose velocity, overlap fraction) แล้วเห็นผลทันที
- Visualize waveform + onsets, pose landmarks
- ทดสอบ video ใหม่ก่อนปล่อยขึ้น production

## Prereq
- รัน `uv sync --all-packages` ที่ repo root
- มี `ffmpeg` ใน PATH
- เริ่ม Jupyter: `cd <repo-root> && uv run jupyter lab`

## 1. Imports + paths

Notebook resolve workspace package ผ่าน `golf-worker` ที่ติดตั้ง editable. ถ้า import ไม่เจอ `worker_app` ตรวจว่ารัน Jupyter จาก repo root

In [ ]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import librosa
import librosa.display

# Workspace packages — must be installed (uv sync --all-packages)
from worker_app.pipeline.audio_onset_librosa import LibrosaAudioOnsetDetector
from worker_app.pipeline.clip_cutter_ffmpeg import FfmpegClipCutter
from worker_app.pipeline.pipeline import Pipeline
from worker_app.pipeline.pose_verifier_mediapipe import MediaPipePoseVerifier

print("python:", sys.version.split()[0])
print("cwd:", os.getcwd())

## 2. Pick a video to analyze

ใส่ path ของ video ที่จะลอง — ไฟล์ในเครื่อง local ไม่ใช่ R2

In [ ]:
VIDEO_PATH = "/Users/user/Downloads/sample.mp4"  # ← แก้เป็น path ของคุณ
PRE_ROLL_SECONDS = 2.0
POST_ROLL_SECONDS = 5.0

assert Path(VIDEO_PATH).exists(), f"video not found: {VIDEO_PATH}"

# สร้าง working dir สำหรับ notebook session — clip output, audio extract
WORKDIR = Path(tempfile.mkdtemp(prefix="golf-nb-"))
AUDIO_PATH = str(WORKDIR / "audio.wav")
CLIPS_DIR = str(WORKDIR / "clips")
Path(CLIPS_DIR).mkdir(exist_ok=True)
print("workdir:", WORKDIR)

## 3. Extract audio (ffmpeg, mono 22.05 kHz เหมือน worker)

In [ ]:
subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i",
        VIDEO_PATH,
        "-vn",
        "-ac",
        "1",
        "-ar",
        "22050",
        AUDIO_PATH,
    ],
    check=True,
    capture_output=True,
)
print(f"wav: {AUDIO_PATH}, size={Path(AUDIO_PATH).stat().st_size/1024:.0f} KB")

## 4. Audio onset detection — ลองหลาย threshold

พารามิเตอร์ที่ tune ได้:
- `delta` (default 0.07) — strength ที่ต้องเกินถึงจะเป็น onset, ยิ่งสูงยิ่ง strict
- `min_separation_seconds` (default 2.0) — onset 2 ตัวห่างกันไม่ถึงเวลานี้ → keep แค่ตัวแรก

In [ ]:
detector = LibrosaAudioOnsetDetector(
    delta=0.07,
    min_separation_seconds=2.0,
)
onsets = detector.detect(AUDIO_PATH)
print(f"detected {len(onsets)} onsets:")
for o in onsets:
    print(f"  t={o.t:6.2f}s  conf={o.confidence:.3f}")

## 5. Visualize waveform + onset markers

In [ ]:
y, sr = librosa.load(AUDIO_PATH, sr=22050, mono=True)
duration = len(y) / sr

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

# Waveform
t_axis = np.arange(len(y)) / sr
axes[0].plot(t_axis, y, lw=0.4, color="steelblue")
axes[0].set_ylabel("amplitude")
axes[0].set_title(f"Waveform — {len(onsets)} onsets detected")
for o in onsets:
    axes[0].axvline(o.t, color="red", alpha=0.4)
    axes[0].text(
        o.t, axes[0].get_ylim()[1] * 0.9, f"{o.t:.1f}", fontsize=7, ha="center", color="red"
    )

# Onset strength envelope
env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=512)
env_t = librosa.frames_to_time(np.arange(len(env)), sr=sr, hop_length=512)
axes[1].plot(env_t, env, color="darkorange", lw=0.8)
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("onset strength")
axes[1].axhline(
    detector._delta * np.max(env),
    color="red",
    ls="--",
    alpha=0.5,
    label=f"delta × max = {detector._delta * np.max(env):.3f}",
)
axes[1].legend(loc="upper right")
for o in onsets:
    axes[1].axvline(o.t, color="red", alpha=0.4)

plt.tight_layout()
plt.show()

## 6. Pose verification — ตรวจแต่ละ onset ว่าเป็น swing จริงไหม

**ครั้งแรกจะ download MediaPipe model ~5MB** — ใช้เวลา ~2s ครั้งเดียว แล้ว cache ที่ `~/.cache/golf-shot-cutter/`

พารามิเตอร์ที่ tune:
- `min_pose_detection_rate` (0-1) — สัดส่วน frame ใน window ที่ต้องเจอคน
- `min_wrist_velocity` (norm coords/frame) — peak wrist movement ขั้นต่ำ

In [ ]:
import time

verifier = MediaPipePoseVerifier(
    min_pose_detection_rate=0.5,
    min_wrist_velocity=0.05,
)

verified = []
rejected = []
for o in onsets:
    t0 = time.time()
    ok = verifier.verify(VIDEO_PATH, o.t)
    elapsed = time.time() - t0
    status = "✅ keep  " if ok else "❌ reject"
    print(f"{status} t={o.t:6.2f}s  conf={o.confidence:.3f}  ({elapsed*1000:.0f}ms)")
    (verified if ok else rejected).append(o)

print(f"\nverified {len(verified)}/{len(onsets)} onsets")

## 7. Visualize pose landmark — เลือก onset 1 ตัวมาดูว่า MediaPipe จับท่าทางได้แบบไหน

In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from worker_app.pipeline.pose_verifier_mediapipe import ensure_pose_model

if not onsets:
    print("no onsets — skip")
else:
    target = onsets[0]  # ← เลือก index ที่อยากเจาะ
    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(target.t * fps))
    ok, frame = cap.read()
    cap.release()

    base_options = mp_python.BaseOptions(model_asset_path=ensure_pose_model())
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.IMAGE,
        num_poses=1,
    )
    landmarker = vision.PoseLandmarker.create_from_options(options)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect(mp_image)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(rgb)
    ax.set_title(f"Pose at t={target.t:.2f}s")
    if result.pose_landmarks:
        lm = result.pose_landmarks[0]
        h, w = rgb.shape[:2]
        # 33 landmarks — สำคัญสำหรับ golf: shoulders (11,12), wrists (15,16), hips (23,24)
        important = {
            11: "L shoulder",
            12: "R shoulder",
            15: "L wrist",
            16: "R wrist",
            23: "L hip",
            24: "R hip",
        }
        for i, p in enumerate(lm):
            color = "lime" if i in important else "cyan"
            ax.plot(p.x * w, p.y * h, "o", color=color, ms=6, alpha=0.8)
            if i in important:
                ax.text(p.x * w + 5, p.y * h, important[i], color="lime", fontsize=8)
    else:
        ax.set_title(f"No pose detected at t={target.t:.2f}s", color="red")
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    landmarker.close()

## 8. Run end-to-end Pipeline (เหมือน worker จริง)

ครั้งนี้ใช้ `Pipeline` เต็มแบบ — onset → pose → dedupe → cut.
ผลออกมาเป็น `ShotCandidate[]` พร้อม clip ใน `CLIPS_DIR/`

In [ ]:
pipeline = Pipeline(
    audio_onset=LibrosaAudioOnsetDetector(
        delta=0.07,
        min_separation_seconds=2.0,
    ),
    pose_verifier=MediaPipePoseVerifier(
        min_pose_detection_rate=0.5,
        min_wrist_velocity=0.05,
    ),
    clip_cutter=FfmpegClipCutter(),
    max_clip_overlap_fraction=0.5,
    pose_max_workers=4,
)

candidates = pipeline.run(
    session_id="nb_demo",
    source_video_path=VIDEO_PATH,
    clips_dir=CLIPS_DIR,
    pre_roll_seconds=PRE_ROLL_SECONDS,
    post_roll_seconds=POST_ROLL_SECONDS,
    audio_path=AUDIO_PATH,
)

print(f"final {len(candidates)} candidates:")
for i, c in enumerate(candidates, 1):
    t_start = max(0, c.t_impact - PRE_ROLL_SECONDS)
    t_end = c.t_impact + POST_ROLL_SECONDS
    print(
        f"  shot {i:02d}: impact={c.t_impact:6.2f}s  window=[{t_start:.1f}–{t_end:.1f}]  conf={c.confidence:.3f}"
    )
    print(f"           clip={c.clip_key}")

## 9. Preview clips ที่ตัดแล้วในหน้า notebook

In [ ]:
from IPython.display import HTML, Video, display

for i, c in enumerate(candidates, 1):
    local = Path(CLIPS_DIR) / f"shot_{i:03d}.mp4"
    if local.exists():
        size_mb = local.stat().st_size / 1024 / 1024
        display(HTML(f"<h4>Shot {i:02d} — impact {c.t_impact:.2f}s ({size_mb:.1f} MB)</h4>"))
        display(Video(str(local), embed=True, width=480))

## 10. Cleanup (optional)

ลบ working dir เมื่อเลิกใช้

In [ ]:
# import shutil
# shutil.rmtree(WORKDIR)
# print('removed', WORKDIR)
print("skipping cleanup — uncomment when done")